# Today vs the new pipeline (scan → rewrite → verify → repair), on real rewrites
This notebook replays the **product's own rewrites** (`both-t5`, detox → formalize, from `outputs/both_detox_formalize.csv`) through the proof-of-concept pipeline in `src/secondthought/pipeline_poc.py`, and judges both with the sendable gate (`src/secondthought/sendable_gate.py`).

- **Today:** the extension shows every rewrite with no checks. A copy of a rude message is even shown as *"This already reads fine"*.
- **New pipeline:** before the rewrite, scan the message into a **manifest** (facts to keep, hostile spans to remove, requests to keep). After the rewrite, check every manifest item, fix single sentences where it's safe, and return warnings for the rest.

**Shadow only.** Nothing in the product calls this code, and no model is loaded: it reads the CSV. Repairs that need no model (putting back a sentence whose facts changed) run for real; retries of still-hostile sentences are shown as *"would retry"*, because the model weights aren't on this machine.

## 1. Run both on the 35 workplace messages

In [ ]:
import csv
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from secondthought.pipeline_poc import compare, manifest, verify

rows = list(csv.DictReader(open(ROOT / "outputs/both_detox_formalize.csv", encoding="utf-8")))
results = pd.DataFrame(compare(rows)).set_index("id")
pd.set_option("display.max_colwidth", None)
print(f"{len(results)} messages, model: {rows[0]['model_info'].split('/')[-1]}")

## 2. Step 1 on one message: the manifest
What the scan pulls out **before** any rewriting. Each item remembers its sentence, so a failure can be traced to the exact sentence that needs fixing.

In [ ]:
m = manifest(rows[0]["original_text"], tone=float(rows[0]["toxicity_original"]))
display(Markdown(f"> {rows[0]['original_text']}"))
display(pd.DataFrame(
    [{"column": "facts (keep exactly)", "sentence": f["sentence"] + 1, "item": f"{f['value']} ({f['kind']})"} for f in m["facts"]]
    + [{"column": "hostile (remove)", "sentence": h["sentence"] + 1, "item": h["text"]} for h in m["hostile"]]
    + [{"column": "requests (keep)", "sentence": r["sentence"] + 1, "item": f"action={r['action']}, deadline={r['deadline']}"} for r in m["requests"]]
))
print("polite (skip the model)?", m["polite"], "| whole-message Detoxify:", m["tone"])

Then step 3 checks the product's rewrite against that manifest. "damn" is gone, but "is a mess" is still there:

In [ ]:
display(Markdown(f"> {rows[0]['rewritten_text']}"))
display(pd.DataFrame(verify(m, rows[0]["rewritten_text"], float(rows[0]["toxicity_rewritten"]))))

## 3. The headline: what the user is shown
**Bad** means the sendable gate fails the product's rewrite.
- **bad today → fixed:** the pipeline repaired it, and the gate now passes it.
- **bad today → now flagged:** the pipeline couldn't fix it offline, so the user would see it **with a warning** instead of silently.
- **bad today → still silent:** the gate fails it but the pipeline noticed nothing (should be 0).
- **gate passes → extra flag:** the pipeline flagged something the gate doesn't check (e.g. a dropped deadline). Each one needs a human look: a real catch, or a false alarm.

In [ ]:
order = ["bad today -> fixed", "bad today -> now flagged", "bad today -> still silent",
         "gate passes -> extra flag (check by hand)", "good, no flag", "skipped (polite)"]
counts = results["outcome"].value_counts().reindex(order, fill_value=0)
display(counts.rename("messages").to_frame())
bad = int((results["today_gate"] != "sendable").sum())
display(Markdown(
    f"**Today:** {bad} of {len(results)} rewrites fail the gate, and **all {bad} reach the user with no warning.**  \n"
    f"**New pipeline:** {counts.iloc[0]} fixed, {counts.iloc[1]} shown with a warning, {counts.iloc[2]} silent."
))

## 4. Fixed: a sentence put back because a fact changed
The formalizer turned `2.5%` into `2,5%` and `4`/`9` into `four`/`nine`. The sentence holding the facts had no hostility, so the pipeline restores the original sentence and keeps the detoxified one.

In [ ]:
fixed = results[results["outcome"] == "bad today -> fixed"]
display(fixed[["original_text", "today_text", "today_gate", "poc_text", "poc_actions", "poc_gate"]])

## 5. Flagged: what the user would now be warned about
None of these can be fixed without the model: the leftover hostility is inside a sentence that **must not** be put back (it is the rude one). With the model loaded, `run(..., retry=...)` retries that sentence once; offline, the rewrite is returned with the warning.

Two of them are **copies**: today the extension shows *"This already reads fine. No changes suggested."* for a message containing "is a joke" and "screwing around". The pipeline blocks those.

In [ ]:
flagged = results[results["outcome"] == "bad today -> now flagged"]
display(flagged[["original_text", "today_text", "today_user_sees", "poc_status", "poc_warnings", "poc_actions"]])

## 6. What every message looked like through the pipeline
The three manifest columns per message, the pipeline's verdict and anything it would tell the user.

In [ ]:
display(results[["facts", "hostile", "requests", "poc_status", "poc_warnings"]])

## 7. What this does **not** show yet
- **The pipeline is graded by the gate it shares checks with.** "Flagged" and "fixed" are partly true by construction. This shows the plumbing works and what the user would see. It does not prove the rewrites are better: that needs the blind human A/B in `new_proposed_eval.md`.
- **Misses are shared too.** Messages where both the gate and the pipeline are wrong show up as "good, no flag":
  - `input_003` "I am sick of your excuses" (rude without a listed word),
  - `input_011` "a complete **show**" (broken phrase after deleting "shitshow"),
  - `input_018`/`032`/`033` "being **unsuitable**" (clumsy word).
  These need the workplace hostility score and the broken-phrase check.
- **No retries ran.** The model isn't on this machine. 7 of the 9 flagged rewrites would get one retry of the hostile sentence (the other two are a tone-only flag and a deleted sentence); how many that fixes is the main open number.
- **Requests never fired.** The product kept every detected action and deadline on this set, so the request check found nothing. It is also a rough heuristic: it misses "Stop being such a jerk to Nina" and "Quit screwing around…" as requests, and names (`Leah`, dropped in `input_015`) are not tracked as facts.
- **Found along the way, in existing code:** the fact extractor ignores a number that ends a sentence (`NUMBER_PATTERN` ends with `(?![\w.])`), so "ordered **23**.", "Suite **21**." and "order **8472**." aren't protected by the gate, the metrics, or the product's `facts_guard`.
- **Word lists flatter this set.** They were written while looking at these 35 messages, and the verb list for requests too.